# 8.28 — Text-to-SQL & semantic parsing

Text-to-SQL succeeds only when language, schema, grammar, and execution all agree on the same meaning. Semantic parsing produces executable programs, so grammar masks and denotation checks matter as much as natural-language fluency. Save a copy to Drive to edit.

In [ ]:

import math
import random
import sqlite3
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8025)
np.random.seed(8025)


def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def cosine(a, b):
    a_arr = np.array(a, dtype=float)
    b_arr = np.array(b, dtype=float)
    denom = np.linalg.norm(a_arr) * np.linalg.norm(b_arr)
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)


def accuracy_score(y_true, y_pred):
    correct = sum(int(a == b) for a, b in zip(y_true, y_pred))
    return correct / max(1, len(y_true))


def precision_recall_f1(y_true, y_pred):
    tp = sum(int(a == 1 and b == 1) for a, b in zip(y_true, y_pred))
    fp = sum(int(a == 0 and b == 1) for a, b in zip(y_true, y_pred))
    fn = sum(int(a == 1 and b == 0) for a, b in zip(y_true, y_pred))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return precision, recall, f1

SCHEMA = {
    "sales": ["region", "revenue", "units", "rep_id"],
    "reps": ["rep_id", "name", "language"],
}
TABLE_ROWS = {
    "sales": [
        {"region": "west", "revenue": 10, "units": 2, "rep_id": 1},
        {"region": "east", "revenue": 30, "units": 5, "rep_id": 2},
        {"region": "north", "revenue": 20, "units": 4, "rep_id": 3},
    ],
    "reps": [
        {"rep_id": 1, "name": "Ava", "language": "en"},
        {"rep_id": 2, "name": "Bo", "language": "es"},
        {"rep_id": 3, "name": "Cy", "language": "fr"},
    ],
}


def constrained_softmax(logits, mask):
    logits_arr = np.array(logits, dtype=float)
    mask_arr = np.array(mask, dtype=float)
    valid_logits = logits_arr[mask_arr == 1]
    valid_probs = softmax(valid_logits)
    probs = np.zeros_like(logits_arr, dtype=float)
    probs[mask_arr == 1] = valid_probs
    return probs


def link_schema(question, schema):
    q = question.lower()
    scores = defaultdict(float)
    for table, columns in schema.items():
        for column in columns:
            key = table + "." + column
            if column in q:
                scores[key] += 1.0
            if "sales" in q and column == "revenue":
                scores[key] += 0.9
            if "highest" in q and column in ["revenue", "units"]:
                scores[key] += 0.4
            if "who" in q and column == "name":
                scores[key] += 0.8
            if "language" in q and column == "language":
                scores[key] += 1.0
    if not scores:
        scores["sales.revenue"] = 0.1
    return dict(scores)


def build_query(question, schema, logits, mask):
    probs = constrained_softmax(logits, mask)
    links = link_schema(question, schema)
    best_link = max(links, key=links.get)
    table, column = best_link.split(".")
    q = question.lower()
    if "highest" in q or "max" in q:
        sql = "SELECT MAX(" + column + ") FROM " + table
    elif "total" in q or "sum" in q:
        sql = "SELECT SUM(" + column + ") FROM " + table
    elif "who" in q:
        sql = "SELECT name FROM reps JOIN sales USING (rep_id) ORDER BY revenue DESC LIMIT 1"
    else:
        sql = "SELECT " + column + " FROM " + table + " LIMIT 1"
    return sql, probs, links


def execute_toy_sql(sql):
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("CREATE TABLE sales(region TEXT, revenue INTEGER, units INTEGER, rep_id INTEGER)")
    cur.execute("CREATE TABLE reps(rep_id INTEGER, name TEXT, language TEXT)")
    for row in TABLE_ROWS["sales"]:
        cur.execute("INSERT INTO sales VALUES (?, ?, ?, ?)", (row["region"], row["revenue"], row["units"], row["rep_id"]))
    for row in TABLE_ROWS["reps"]:
        cur.execute("INSERT INTO reps VALUES (?, ?, ?)", (row["rep_id"], row["name"], row["language"]))
    cur.execute(sql)
    result = cur.fetchall()
    conn.close()
    return result


def semantic_parse(question, schema, logits, mask):
    sql, probs, links = build_query(question, schema, logits, mask)
    result = execute_toy_sql(sql)
    return {"sql": sql, "probs": probs, "links": links, "result": result}


def text_to_sql_dataset_ladder():
    d1 = [
        {"question": "highest sales", "gold": [(30,)], "logits": [2, 5, 1, 4], "mask": [1, 0, 1, 0]},
    ]
    d2 = [
        {"question": "highest sales", "gold": [(30,)], "logits": [2, 5, 1, 4], "mask": [1, 0, 1, 0]},
        {"question": "total revenue", "gold": [(60,)], "logits": [1, 2, 3, 0], "mask": [1, 1, 1, 0]},
        {"question": "highest units", "gold": [(5,)], "logits": [2, 1, 3, 0], "mask": [1, 1, 1, 0]},
        {"question": "who has highest sales", "gold": [("Bo",)], "logits": [2, 2, 1, 0], "mask": [1, 1, 1, 0]},
        {"question": "first region", "gold": [("west",)], "logits": [1, 1, 1, 0], "mask": [1, 1, 1, 0]},
    ]
    d3 = d2 + [
        {"question": "highest sales by language", "gold": [(30,)], "logits": [1, 5, 2, 4], "mask": [1, 0, 1, 0]},
        {"question": "sum units", "gold": [(11,)], "logits": [2, 1, 1, 0], "mask": [1, 1, 1, 0]},
    ]
    d4 = d3 + [
        {"question": "who has the maximum revenue", "gold": [("Bo",)], "logits": [2, 3, 1, 0], "mask": [1, 1, 1, 0]},
        {"question": "show language", "gold": [("en",)], "logits": [1, 1, 1, 0], "mask": [1, 1, 1, 0]},
    ]
    d5 = d4 + [
        {"question": "which representative should I call for the highest sales region", "gold": [("Bo",)], "logits": [2, 4, 3, 1], "mask": [1, 0, 1, 0]},
        {"question": "total units in the sales table", "gold": [(11,)], "logits": [2, 2, 2, 0], "mask": [1, 1, 1, 0]},
        {"question": "highest revenue after joining reps", "gold": [(30,)], "logits": [2, 5, 1, 4], "mask": [1, 0, 1, 0]},
    ]
    return [
        {"rung": "D1", "name": "lesson schema and action toy", "items": d1},
        {"rung": "D2", "name": "five clean single-table questions", "items": d2},
        {"rung": "D3", "name": "schema ambiguity and invalid actions", "items": d3},
        {"rung": "D4", "name": "inline SQLite database questions", "items": d4},
        {"rung": "D5", "name": "multi-table aggregation questions", "items": d5},
    ]


## The concept, built once: constrained action probabilities

The lesson masks invalid SQL actions before normalizing:

$$P(a_t=k\mid a_{\lt t},q,\mathcal{S})=rac{m_k e^{z_k}}{\sum_j m_j e^{z_j}},\qquad \operatorname{ExecAcc}=rac{1}{N}\sum_{n=1}^N\mathbb{1}[\operatorname{exec}(\hat y_n)=\operatorname{exec}(y_n)].$$

We implement constrained parsing and assert that the invalid raw logit `5` receives probability `0`.

In [ ]:

parsed = semantic_parse("highest sales", SCHEMA, logits=[2, 5, 1, 4], mask=[1, 0, 1, 0])
probs = parsed["probs"]

assert [round(p, 3) for p in probs] == [0.731, 0.0, 0.269, 0.0]
assert probs[1] == 0.0
assert parsed["result"] == [(30,)]

print("masked probabilities", [round(p, 3) for p in probs])
print("SQL", parsed["sql"])
print("execution result", parsed["result"])


## Schema links, action sequences, execution, and exact-match divergence

The lesson links `sales` to `revenue` with margin `0.7`, uses four planned actions, executes a maximum of `30`, and shows execution accuracy can exceed exact match.

In [ ]:

link_scores = np.array([[0.8, 0.1], [0.9, 0.2]])
margin = link_scores[1, 0] - link_scores[1, 1]
actions = ["SELECT", "AGG_MAX", "COLUMN_revenue", "FROM_sales"]
values = [10, 30, 20]
exact = [1, 0, 0]
execution = [1, 1, 0]

assert abs(margin - 0.7) < 1e-12
assert len(actions) == 4
assert max(values) == 30
assert sum(execution) > sum(exact)

print("schema-link margin", margin)
print("action count", len(actions))
print("max value", max(values))
print("exact vs exec totals", sum(exact), sum(execution))


## The dataset ladder: D1 to D5 synthetic semantic parsing

The ladder grows from a masked action toy to inline SQLite queries with ambiguous schemas, joins, and aggregations.

In [ ]:

ladder = text_to_sql_dataset_ladder()

for rung in ladder:
    questions = [item["question"] for item in rung["items"]]
    print(rung["rung"], rung["name"], "questions=", len(questions))
    print("sample:", questions[0])


## Run the same semantic parser across D1-D5

The metric is execution accuracy: does the predicted SQL return the reference denotation?

In [ ]:

results = []

for rung in ladder:
    y_true = []
    y_pred = []
    masks = []
    links = []
    for item in rung["items"]:
        parsed = semantic_parse(item["question"], SCHEMA, item["logits"], item["mask"])
        y_true.append(item["gold"])
        y_pred.append(parsed["result"])
        masks.append(parsed["probs"].tolist())
        links.append(parsed["links"])
    exec_acc = accuracy_score(y_true, y_pred)
    results.append({"rung": rung["rung"], "exec_accuracy": exec_acc, "masks": masks, "links": links})

print("rung exec_accuracy")
for row in results:
    print(row["rung"], round(row["exec_accuracy"], 3))


## Results visualization: schema/action panels and metric curve

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

for ax, rung, row in zip(axes, ladder, results):
    matrix = np.array(row["masks"][:5], dtype=float)
    ax.imshow(matrix, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_title(rung["rung"])
    ax.set_xticks(range(4))
    ax.set_xticklabels(["a0", "a1", "a2", "a3"])
    ax.set_yticks([])

plt.suptitle("Action-mask probability panels by rung")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot([row["rung"] for row in results], [row["exec_accuracy"] for row in results], marker="o")
plt.ylim(0, 1.05)
plt.ylabel("execution accuracy")
plt.xlabel("rung")
plt.title("Exec accuracy vs schema and query complexity")
plt.grid(True)
plt.show()


## Pitfall on D5: skipped schema linking and stale masks

A parser can produce valid SQL that selects the wrong column, and a stale grammar mask can let invalid logits drive decoding. The fix is explicit schema links, constrained decoding, and execution checks.

In [ ]:

d5 = ladder[-1]["items"]
y_true = [item["gold"] for item in d5]
linked_results = []
stale_results = []

for item in d5:
    parsed = semantic_parse(item["question"], SCHEMA, item["logits"], item["mask"])
    linked_results.append(parsed["result"])
    stale_mask = [1, 1, 1, 1]
    stale = semantic_parse(item["question"], SCHEMA, item["logits"], stale_mask)
    stale_results.append(stale["result"])

linked_acc = accuracy_score(y_true, linked_results)
stale_acc = accuracy_score(y_true, stale_results)

print("linked constrained exec accuracy", round(linked_acc, 3))
print("stale-mask exec accuracy", round(stale_acc, 3))
assert linked_acc >= stale_acc


## Evaluate it + Practice

- Metric: execution accuracy, alongside exact-match only as a diagnostic.
- Sanity check: invalid raw logit `5` must receive probability `0` under the grammar mask.
- Ablation: skip schema linking or use a stale all-ones mask and watch execution accuracy drop.
- Failure signals: syntactically valid SQL with wrong denotation, wrong aggregation, or wrong table.

Practice: add a `WHERE region = 'east'` question and extend the grammar actions.

Practice: compare exact-match and execution accuracy for two equivalent SQL strings.

Practice: create a deliberately wrong schema link from `sales` to `region` and execute the result.